# Winner 공개 예측: CAT + LGB + XGB 평균과 UID PP
[Part 2](https://www.kaggle.com/competitions/ieee-fraud-detection/writeups/fraudsquad-1st-place-solution-part-2)의 동일 가중치 앙상블을 공개된 best-single 예측 CSV로 구성한다.
학습하는 노트북이 아니다. CAT/LGB/XGB를 새로 학습한 결과와 구분하며 `winner_xgb`의 smoke 예측을 섞지 않는다.
추가 입력은 [우승자가 공개한 예측/UID 데이터셋 version 2](https://www.kaggle.com/datasets/kyakovlev/ieee-submissions-and-uids)이다.
`lgbm_meta_model.csv`는 test 예측만 있어 스태킹 학습용 OOF를 제공하지 않는다. 스태킹 모델을 임의로 만들지 않는다.

## 이 노트북을 읽는 방법

코드를 실행하기 전에 바로 위의 설명을 읽어보세요. **어떤 질문을 푸는지 → 작은 예시로 계산 → 실제 코드의 변수와 연결 → 출력 해석** 순서로 설명합니다.
코드 아래의 관찰은 이미 저장된 실행 결과를 읽는 안내입니다. '해볼 실험'은 아직 실행하지 않은 제안이며, 실제 결과와 구분했습니다.

처음 읽을 때 함수 이름을 모두 외울 필요는 없습니다. 새 피처를 만날 때마다 **'이 숫자는 무엇을 요약하며, 예측할 때도 알 수 있는가?'**를 물어보세요.
EDA에서 찾은 차이가 모델 성능 개선을 뜻하지는 않습니다. 실험 노트북에서는 **검증 데이터를 정한 뒤 구성 요소 하나씩 비교**해야 개선의 근거를 얻습니다.

처음에는 `01_eda_report` → GitHub `baseline`을 읽고, 이후 `02_winner_eda` → `winner_xgb` → `winner_lgbm` → `winner_catboost` → `winner_blend`로 이어가세요.
우승자 공개 baseline과 최종 우승 제출 전체는 구분합니다.

설명을 보강하면서 학습 코드·기존 표·그래프·실행 범위를 유지했습니다. 이 노트북의 **저장된 출력**과 설정의 **다음 실행 기본값**이 다를 수 있으므로 첫 실행 범위와 metrics를 먼저 확인하세요.


### 처음 만나는 용어는 여기서 잠깐 확인하세요

| 용어 | 여기서 뜻하는 것 |
|---|---|
| 피처(feature) | 모델에 입력할 거래의 정보. 원본 열과 새로 계산한 열 모두 포함 |
| NaN / 결측 | 값이 관측되지 않음. 실제 숫자 0과 다른 상태 |
| fold / validation | 교차검증의 한 분할 / 그 분할에서 평가용으로 제외한 데이터 |
| OOF | 각 train 행을 그 행 없이 학습한 모델로 예측해서 모은 값 |
| smoke | 전체 학습 전에 축소 데이터·rounds로 실행 흐름을 확인하는 실험 |
| leaf | tree에서 조건을 따라 내려간 끝의 구역. 그 구역에 들어온 행에 같은 보정을 줌 |

**제거 실험(ablation)**은 피처나 기법 하나를 뺀 모델을 같은 검증에서 비교하는 방법입니다. '있을 때 좋았으니 도움이 된다'에서 한 걸음 더 나아가 실제 기여를 확인하려는 실험입니다.

### 이 노트북의 질문: 여러 예측을 평균내면 무엇이 바뀔까요?

앞 실험의 30,000행 smoke 모델을 합치는 노트북이 아닙니다. 저자가 공개한 dataset **version 2**의 best-single CAT/LGB/XGB test 확률을 읽습니다.
모델 학습 없이 전체 test 506,691건에 대한 평균과 UID post-processing(PP)을 재현합니다. train 원본 CSV의 라벨은 PP에서 이미 아는 정답으로 사용합니다.

**먼저 거래를 맞춥니다.** 파일 행 번호가 같아도 같은 거래라는 보장은 없습니다. 각 입력의 ID가 유일하고 sample과 ID 집합이 같은지 검사한 뒤 `reindex(sample.index)`로 정렬합니다.
잘못된 행끼리 평균하면 겉보기 확률 범위는 정상이어도 엉뚱한 거래의 예측이 됩니다. 0~1 검사와 ID 검사는 서로 다른 오류를 잡습니다.

**그다음 행별 평균입니다.** 한 거래의 CAT/LGB/XGB 확률이 [.2,.4,.6]이면 `mean(axis=1)`은 .4입니다.
모두 같은 방향으로 틀리면 평균도 틀립니다. 서로 다른 잡음이 섞일 때 흔들림이 줄 수 있습니다. 이를 분산으로 쓰면
$$\mathrm{Var}\!\left(\frac{e_1+e_2+e_3}{3}\right)=\frac{\sum_i\mathrm{Var}(e_i)+2\sum_{i<j}\mathrm{Cov}(e_i,e_j)}{9}.$$
$e_i$는 설명용 예측 오차입니다. 오차가 매우 비슷하면 covariance가 커져 평균의 이점이 줄어듭니다. 평균이 편향이나 AUC를 항상 개선한다는 보장은 없습니다.

출력 `inputs.corr()`는 **예측 확률끼리**의 Pearson 상관입니다. test 정답이 없으므로 오차 상관을 직접 측정한 것이 아닙니다.
상관이 낮다고 반드시 유용한 새 정보가 생기는 것도 아닙니다. 순전히 나쁜 모델의 예측도 상관은 낮을 수 있습니다.
또한 AUC는 단일 모델의 순서에 관심이 있지만 평균은 확률의 크기도 사용합니다. 같은 AUC를 가진 두 모델이라도 확률 범위가 다르면 blend 기여가 달라질 수 있습니다.

### UID PP는 평균의 단위를 '같은 거래'에서 '같은 추정 고객'으로 바꿉니다

같은 UID의 test 예측 [.2,.8]과 알려진 train 정상 라벨 0이 한 그룹이라면 평균 `[.2,.8,0]`=1/3을 test 두 건에 적용합니다.
고객 내 정답이 일관되고 UID 복원이 맞을 때 잡음을 줄일 수 있습니다. 혼합 라벨 고객이나 잘못 합친 UID에서는 올바른 개별 예측도 망칠 수 있습니다.
`postprocess`는 원문과 같이 v4→v1 순서로 실행합니다. 첫 단계에서 train 값도 갱신하므로 두 단계는 서로 독립적인 평균이 아닙니다.
UID가 없는 거래는 기존 예측을 유지합니다. identity 부재, UID 미복원, train과 UID 미겹침도 같은 현상을 뜻하지 않습니다.

**왜 이 PP를 OOF에 그대로 쓰면 안 될까요?** Validation 행의 정답을 known label에 넣으면 그 행의 정답이 자기 예측에 들어갑니다.
PP 효과를 검증하려면 각 fold의 fit 라벨만 사용한 PP가 필요합니다. 여기서는 test PP만 실행하고 그 별도 OOF 실험은 하지 않았습니다.

평균 blend는 학습된 가중치가 없는 단순 기법입니다. Stacking은 기본 모델의 OOF 예측을 입력으로 두 번째 모델을 학습하는 기법입니다.
학습 행에 대해 이미 정답을 보고 만든 in-sample 예측으로 stacking을 학습하면 과적합 위험이 큽니다. 이 노트북은 공개 최종 우승 stacking 전체를 복원한 코드가 아닙니다.

In [1]:
import json, logging, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'source.json').exists())
sys.path.insert(0, str(ROOT))
from data.loader import get_data_dir
from data.winner import RELEASED_DATASET, get_released_dir, postprocess
raw, released = get_data_dir(), get_released_dir()
OUT = ROOT / 'experiments/winner_blend/outputs/released_v2'
OUT.mkdir(parents=True, exist_ok=True)
started = time.perf_counter()
sample = pd.read_csv(raw / 'sample_submission.csv').set_index('TransactionID')
labels = pd.read_csv(raw / 'train_transaction.csv', usecols=['TransactionID', 'isFraud']).set_index('TransactionID').isFraud
names = ['catboost_best_single.csv', 'lgbm_best_single.csv', 'xgb_best_single.csv']
inputs = {}
for name in names:
    frame = pd.read_csv(released / name).set_index('TransactionID')
    assert frame.index.is_unique and set(frame.index) == set(sample.index)
    inputs[name] = frame.isFraud.reindex(sample.index)
inputs = pd.DataFrame(inputs)
assert np.isfinite(inputs).all().all() and inputs.ge(0).all().all() and inputs.le(1).all().all()
blended = inputs.mean(axis=1).rename('isFraud')
processed, coverage = postprocess(labels, blended, released)
blended.to_csv(OUT / 'submission_equal_mean.csv')
processed.to_csv(OUT / 'submission_equal_mean_pp.csv')
display(inputs.corr())
display(pd.DataFrame(coverage))

,catboost_best_single.csv,lgbm_best_single.csv,xgb_best_single.csv
catboost_best_single.csv,1.000000,0.957390,0.943452
lgbm_best_single.csv,0.957390,1.000000,0.983161
xgb_best_single.csv,0.943452,0.983161,1.000000


,file,test_coverage
0,uids_v4_no_multiuid_cleaning..csv,0.693350
1,uids_v1_no_multiuid_cleaning.csv,0.719484


### 아래 출력에서 찾아볼 것
상관 표의 대각선 1은 자기 자신과의 상관입니다. 비대각선에서 어느 쌍의 예측이 더 비슷한지 보세요. 이는 test 성능 순위가 아닙니다.
PP의 test UID 커버리지는 v4 약 69.33%, v1 약 71.95%입니다. 두 집합이 겹치므로 합산해 전체 커버리지로 읽지 않습니다.
train에서 같은 UID를 본 비율은 이 커버리지와 다른 지표입니다. 출력의 이름과 분모를 함께 확인하세요.

공개된 `final_model_blend.csv`와의 차이를 측정한다. 작성자는 최종 blend/stack+PP라고 설명했지만, 정확한 가중치와 메타모델 학습 설정은 이 9개 자료에서 복원할 수 없다.
따라서 평균+XGB 공개 PP 결과를 최종 우승 제출 그 자체라고 부르지 않는다. test 정답은 공개되지 않았으므로 LB/AUC를 계산하지 않는다.

### 공개 최종 제출과의 차이는 '성능 차이'가 아닙니다

`final_model_blend.csv`와 우리 equal-mean+PP의 거래별 확률 차이를 `delta`로 계산합니다.
예를 들어 우리 .4, 공개 최종 .5이면 delta=-.1이고 절댓값은 .1입니다. `mean_absolute`는 이 확률 차이의 평균, `max_absolute`는 가장 큰 한 건의 차이입니다.
**정답과 비교한 예측 오차가 아닙니다.** 두 제출이 다르다는 것을 확인하는 수치입니다.

`auc=None`인 이유는 test 라벨이 공개되지 않았기 때문입니다. AUC가 0이라는 뜻도, 제출이 나쁘다는 뜻도 아닙니다.
이 파일만으로 'PP가 AUC를 올렸다'거나 '공개 최종과 같은 우승 점수'라고 말할 수 없습니다. 공개 최종의 모든 학습/가중치를 복원한 것도 아닙니다.

세 입력을 1/3씩 평균내는 것은 여기서 명시한 재현 범위입니다. 따로 보존한 저자의 internal blend에는 다른 입력과 합산 코드가 있어 이 equal-mean과 동일한 절차로 읽지 않습니다.
자세한 파일·버전·원문 연결은 `references/winner/README.md`에 있습니다.

**아직 실행하지 않은 학습용 비교:** 동일한 validation에서 기본 모델별 raw OOF → equal mean OOF → fold별 fit 라벨만 사용한 PP를 비교합니다.
가중치를 고를 때도 별도 평가셋이 필요합니다. 현재 공개 best-single 파일은 test 예측이므로 그것만으로 가중치를 검증할 수 없습니다.

In [2]:
final = pd.read_csv(released / 'final_model_blend.csv').set_index('TransactionID').isFraud
assert final.index.is_unique and set(final.index) == set(sample.index)
delta = processed - final.reindex(sample.index)
metrics = {'scope': 'released predictions only; no model training', 'dataset': RELEASED_DATASET,
           'rows': len(sample), 'inputs': names, 'weights': [1 / 3] * 3, 'pp_coverage': coverage,
           'difference_to_released_final': {'mean_absolute': float(delta.abs().mean()), 'max_absolute': float(delta.abs().max())},
           'auc': None, 'elapsed_seconds': time.perf_counter() - started}
(OUT / 'metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
(OUT / 'run.log').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
np.save(OUT / 'pred_test.npy', processed.to_numpy())
display(metrics)

{'scope': 'released predictions only; no model training',
 'dataset': 'kyakovlev/ieee-submissions-and-uids/versions/2',
 'rows': 506691,
 'inputs': ['catboost_best_single.csv',
  'lgbm_best_single.csv',
  'xgb_best_single.csv'],
 'weights': [0.3333333333333333, 0.3333333333333333, 0.3333333333333333],
 'pp_coverage': [{'file': 'uids_v4_no_multiuid_cleaning..csv',
   'test_coverage': 0.693349595710206},
  {'file': 'uids_v1_no_multiuid_cleaning.csv',
   'test_coverage': 0.7194838668932347}],
 'difference_to_released_final': {'mean_absolute': 0.003512858292503631,
  'max_absolute': 0.3330952415184279},
 'auc': None,
 'elapsed_seconds': 3.8335561002604663}

### 현재 실행에서 확인한 사실
전체 506,691건을 처리했고 공개 최종 예측과의 평균 절대 차이는 약 .003513, 최대 차이는 .333095입니다.
대체로 비슷한 예측이어도 일부 거래의 차이가 클 수 있습니다. 그 차이가 개선인지 악화인지는 test 정답 없이 판단할 수 없습니다.
**스스로 설명해보세요:** 거래별 모델 평균과 UID별 PP 평균은 어떤 축으로 묶나요? Train 라벨을 쓸 수 있는 test PP와 validation 정답을 쓰면 안 되는 OOF PP는 어떻게 다른가요?